# Week 4 — Financial Forecasting & Predictive Modeling

This notebook uses the Week 2 cleaned panel for TCS, Infosys, HCLTech and Wipro.

In [ ]:
import pandas as pd
from statsmodels.tsa.holtwinters import Holt
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

df=pd.read_csv('../data/cleaned/quarterly_cleaned.csv')
df.head()

## Validation design

The final three observed quarters are held out. Models are fitted only on earlier quarters, then evaluated on the unseen holdout. This preserves temporal ordering and avoids using future information during evaluation.

In [ ]:
# Example: TCS revenue
s=df[df.Company=='TCS'].sort_values('Period End')['Revenue (INR Cr)'].to_numpy()
train,test=s[:-3],s[-3:]
print('Train:',train)
print('Test:',test)

## Candidate models

Naive baseline, linear drift, damped Holt exponential smoothing, ARIMA(1,1,0), and AutoReg(1) are compared. Model selection is based on holdout MAE, with RMSE and MAPE reported as additional diagnostics.

In [ ]:
def metric(a,p):
    return {'MAE':mean_absolute_error(a,p),'RMSE':mean_squared_error(a,p)**0.5,'MAPE (%)':mean_absolute_percentage_error(a,p)*100}

models={
'Naive':lambda y,h:[y[-1]]*h,
'Drift':lambda y,h:__import__('numpy').polyval(__import__('numpy').polyfit(range(len(y)),y,1),range(len(y),len(y)+h)),
'Holt damped':lambda y,h:Holt(y,damped_trend=True,initialization_method='estimated').fit().forecast(h),
'ARIMA(1,1,0)':lambda y,h:ARIMA(y,order=(1,1,0),trend='t').fit().forecast(h),
'AutoReg(1)':lambda y,h:AutoReg(y,lags=1,trend='ct',old_names=False).fit().predict(start=len(y),end=len(y)+h-1)}
for name,f in models.items():
    p=f(train,3)
    print(name, metric(test,p))

## Final forecast

After selecting the lowest holdout-MAE model for each target/company pair, the selected model is refit using all 13 observed quarters and used to forecast Q2 FY27 through Q1 FY28.